# Lab Week 7-8 NLP

#  Open Source LLMs — Lab Notebook
## Fine-Tuning · Text Generation · Question Answering

**Prerequisites:** Python 3.8+, ~4 GB free disk space  
**Estimated time:** 90–120 minutes  

---
### Learning Objectives
By the end of this lab you will be able to:
1. Load and run inference with small open-source LLMs
2. Compare text-generation decoding strategies
3. Build extractive and generative QA pipelines
4. Fine-tune a causal LM on custom data using the HuggingFace `Trainer`
5. Apply LoRA for parameter-efficient fine-tuning

---
### Models used (all < 500 MB, CPU-friendly)
| Model | Size | Task |
|---|---|---|
| `distilgpt2` | ~330 MB | Causal LM / text generation |
| `google/flan-t5-small` | ~300 MB | Seq2Seq / instruction QA |
| `deepset/roberta-base-squad2` | ~500 MB | Extractive QA |


##  Setup — Install Dependencies

In [3]:
# Run this cell once to install all required packages
!pip install transformers datasets peft accelerate torch sentencepiece -q

##  Imports

In [4]:
import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoModelForQuestionAnswering,
    pipeline,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {'GPU ✓' if torch.cuda.is_available() else 'CPU'}")

PyTorch  : 2.10.0
Device   : CPU


## Section 1 — Load a Model and Tokenizer

The HuggingFace `transformers` library provides a unified API for hundreds of models.

**Key classes:**
- `AutoTokenizer.from_pretrained(name)` — loads the correct tokenizer for a model
- `AutoModelForCausalLM.from_pretrained(name)` — loads a causal language model (GPT family)

> 💡 **Causal LM** = predicts the *next* token given all previous tokens. Used for text generation.


In [5]:
MODEL_NAME = "distilgpt2"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

# GPT-2 has no padding token — assign eos_token as pad
tokenizer.pad_token = tokenizer.eos_token

print(f"Model    : {MODEL_NAME}")
print(f"Params   : {model.num_parameters():,}")
print(f"Vocab sz : {tokenizer.vocab_size:,}")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 10917.80it/s]


Model    : distilgpt2
Params   : 81,912,576
Vocab sz : 50,257


### 1b — Tokenisation

Tokenisation converts raw text into integer IDs that the model understands.

In [6]:
text = "Transformers are amazing for NLP tasks!"

# Encode text → token IDs
token_ids = tokenizer.encode(text)
print("Token IDs :", token_ids)

# Decode back to string
decoded = tokenizer.decode(token_ids)
print("Decoded   :", decoded)

# Show individual tokens
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print("Tokens    :", tokens)

Token IDs : [41762, 364, 389, 4998, 329, 399, 19930, 8861, 0]
Decoded   : Transformers are amazing for NLP tasks!
Tokens    : ['Transform', 'ers', 'Ġare', 'Ġamazing', 'Ġfor', 'ĠN', 'LP', 'Ġtasks', '!']


---
## Section 2 — Text Generation & Decoding Strategies

`model.generate()` supports multiple decoding strategies:

| Strategy | `do_sample` | Extra args | Character |
|---|---|---|---|
| **Greedy** | `False` | — | Deterministic, repetitive |
| **Beam search** | `False` | `num_beams=5` | Better quality, slower |
| **Temperature** | `True` | `temperature=0.8` | Creative / risky |
| **Top-p (nucleus)** | `True` | `top_p=0.9, temperature=0.7` | Best diversity/quality balance |


### 2a — Helper function

In [8]:
def generate_text(prompt: str, max_new_tokens: int = 80,
                  strategy: str = "greedy") -> str:
    """Generate text continuation from a prompt using various strategies."""
    inputs = tokenizer(prompt, return_tensors="pt")

    if strategy == "greedy":
        gen_kwargs = dict(do_sample=False)
    elif strategy == "beam":
        gen_kwargs = dict(do_sample=False, num_beams=5, early_stopping=True)
    elif strategy == "temperature":
        gen_kwargs = dict(do_sample=True, temperature=0.8)
    elif strategy == "top_p":
        gen_kwargs = dict(do_sample=True, top_p=0.92, temperature=0.7)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            **gen_kwargs,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Quick test
print(generate_text("Deep learning is", strategy="greedy").strip())


a great way to learn about the world and how to learn about the world.


### 2b — Compare all strategies on the same prompt

In [9]:
PROMPT = "Artificial intelligence is transforming the world by"

for strategy in ["greedy", "beam", "temperature", "top_p"]:
    result = generate_text(PROMPT, strategy=strategy)
    print(f"[{strategy:12s}] {result[:100]}".strip())
    print()

[greedy      ]  creating a new kind of artificial intelligence.

[beam        ]  creating artificial intelligence.

[temperature ]  altering the environment and creating new things. Artificial intelligence is a new way to improve o

[top_p       ]  creating a new type of software that can be used to solve problems in everyday life.



---
## Section 3 — Question Answering with FLAN-T5

`google/flan-t5-small` is a Seq2Seq model fine-tuned to follow natural language instructions.
It can answer open-domain questions, translate, summarise, and classify — all in one model.

> 💡 **Seq2Seq** = Encoder processes input, Decoder generates output token by token.


In [10]:
T5_NAME = "google/flan-t5-small"
tok_t5  = AutoTokenizer.from_pretrained(T5_NAME)
mdl_t5  = AutoModelForSeq2SeqLM.from_pretrained(T5_NAME)
mdl_t5.eval()
print(f"Loaded {T5_NAME}  |  {mdl_t5.num_parameters():,} parameters")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 5802.05it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded google/flan-t5-small  |  76,961,152 parameters


### 3b — Inference function

In [11]:
def ask_flan_t5(instruction: str, max_new_tokens: int = 100) -> str:
    """Send an instruction to FLAN-T5 and return the generated answer."""
    inputs = tok_t5(instruction, return_tensors="pt", truncation=True)
    with torch.no_grad():
        out = mdl_t5.generate(**inputs, max_new_tokens=max_new_tokens)
    return tok_t5.decode(out[0], skip_special_tokens=True)


# Demo
questions = [
    "What is the capital of Japan?",
    "Translate to French: The weather is beautiful today.",
    "Classify the sentiment of: I absolutely loved the movie!",
    "What are two advantages of renewable energy?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_flan_t5(q)}\n")


Q: What is the capital of Japan?
A: yokohama

Q: Translate to French: The weather is beautiful today.
A: La météo est magnifique aujourd'hui.

Q: Classify the sentiment of: I absolutely loved the movie!
A: positive

Q: What are two advantages of renewable energy?
A: Renewable energy is a renewable resource



---
## Section 4 — Extractive Question Answering

Extractive QA finds the answer *span* inside a provided context.  
Model: `deepset/roberta-base-squad2` — fine-tuned on SQuAD2 dataset.

```
Context : "The Transformer was introduced in 2017 by Vaswani et al."
Question: "Who introduced the Transformer?"
Answer  : "Vaswani et al."  ← a span extracted from the context
```

In [14]:
qa_pipe = pipeline("question-answering", model="deepset/roberta-base-squad2")
print("QA pipeline ready")

context = """
The Transformer architecture was introduced in the landmark 2017 paper
'Attention Is All You Need' by Vaswani et al. at Google Brain. Unlike
recurrent neural networks, Transformers process all tokens in parallel
using self-attention, which allows each token to attend to every other
token in the sequence. BERT, GPT, and T5 are all based on Transformers.
"""

qa_pairs = [
    "When was the Transformer architecture introduced?",
    "What mechanism do Transformers use to process tokens?",
    "Which models are based on the Transformer architecture?",
]

for question in qa_pairs:
    result = qa_pipe(question=question, context=context)
    print(f"Q: {question}")
    print(f"A: {result['answer']}  (confidence: {result['score']:.2%})\n")


KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

Section 5 — Fine-Tuning a Causal LM on Custom Data
Fine-tuning adapts a pre-trained model to your domain by continuing training on new data.

Steps:

Prepare a dataset of domain texts
Tokenise (labels = input_ids for causal LM)
Configure TrainingArguments
Train with Trainer
Save and reload

### 5a — Prepare the dataset

In [15]:
CUSTOM_TEXTS = [
    "Machine learning models learn patterns from data to make predictions.",
    "Neural networks consist of layers of interconnected nodes called neurons.",
    "Gradient descent minimises the loss function by updating model weights iteratively.",
    "Overfitting occurs when a model performs well on training data but poorly on new data.",
    "Regularisation techniques like dropout help prevent overfitting in deep networks.",
    "Transfer learning allows models to apply knowledge from one task to another.",
    "Attention mechanisms allow models to focus on the most relevant parts of the input.",
    "BERT uses bidirectional context to build rich language representations.",
    "The learning rate controls the size of update steps during gradient descent.",
    "Batch normalisation stabilises training by normalising layer activations.",
    "Convolutional neural networks are highly effective for image recognition tasks.",
    "Large language models are pre-trained on massive text corpora before fine-tuning.",
    "Tokenisation converts raw text into numerical tokens the model can process.",
    "An embedding layer maps discrete tokens to continuous high-dimensional vectors.",
    "The softmax function converts raw logits into a probability distribution over tokens.",
]

# Tokeniser for fine-tuning
FT_NAME  = "distilgpt2"
ft_tok   = AutoTokenizer.from_pretrained(FT_NAME)
ft_tok.pad_token = ft_tok.eos_token

def tokenize_fn(examples):
    tokens = ft_tok(examples["text"], truncation=True, max_length=64, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

raw_ds  = Dataset.from_dict({"text": CUSTOM_TEXTS})
tok_ds  = raw_ds.map(tokenize_fn, batched=True)
tok_ds.set_format("torch")

print(f"Dataset : {len(tok_ds)} samples")
print(f"Columns : {tok_ds.column_names}")


Map: 100%|██████████| 15/15 [00:00<00:00, 986.26 examples/s]

Dataset : 15 samples
Columns : ['text', 'input_ids', 'attention_mask', 'labels']


### 5b — Configure and run training

In [16]:
ft_model = AutoModelForCausalLM.from_pretrained(FT_NAME)

training_args = TrainingArguments(
    output_dir          = "./finetuned_llm",
    num_train_epochs    = 3,
    per_device_train_batch_size = 4,
    learning_rate       = 5e-5,
    weight_decay        = 0.01,
    warmup_steps        = 10,
    logging_steps       = 5,
    save_strategy       = "epoch",
    report_to           = "none",
    fp16                = torch.cuda.is_available(),
    dataloader_pin_memory = False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=ft_tok, mlm=False)

trainer = Trainer(
    model         = ft_model,
    args          = training_args,
    train_dataset = tok_ds,
    data_collator = data_collator
)

trainer.train()
print("\n✓ Training complete!")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7740.82it/s]
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,5.272405
10,4.772840


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.15it/s]



✓ Training complete!


### 5c — Save and reload


In [20]:
SAVE_PATH = "./finetuned_llm/final"
trainer.save_model(SAVE_PATH)
ft_tok.save_pretrained(SAVE_PATH)
print(f"Saved to {SAVE_PATH}")

# Reload
ft_model_loaded = AutoModelForCausalLM.from_pretrained(SAVE_PATH)
ft_tok_loaded   = AutoTokenizer.from_pretrained(SAVE_PATH)
ft_model_loaded.eval()

def generate_ft(prompt, max_new_tokens=60):
    inputs = ft_tok_loaded(prompt, return_tensors="pt")
    with torch.no_grad():
        out = ft_model_loaded.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, top_p=0.9, temperature=0.8,
            pad_token_id=ft_tok_loaded.eos_token_id,
        )
    new_ids = out[0][inputs["input_ids"].shape[1]:]
    return ft_tok_loaded.decode(new_ids, skip_special_tokens=True)

for prompt in ["Neural networks consist of", "The learning rate controls"]:
    print(f"PROMPT : {prompt}")
    print(f"OUTPUT : {generate_ft(prompt)}\n".strip())


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.16it/s]


Saved to ./finetuned_llm/final


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 14657.31it/s]


PROMPT : Neural networks consist of
OUTPUT :  axial networks which are associated with the same axial network. Each axial network consists of axial networks which are associated with the same axial network. Each axial network consists of axial networks which are associated with the same axial network.





The axial
PROMPT : The learning rate controls
OUTPUT :  the neural networks to identify new information.


---
## Section 6 — LoRA: Parameter-Efficient Fine-Tuning

**Problem:** Fine-tuning large models requires updating hundreds of millions of weights → slow and memory-hungry.  
**Solution:** LoRA (Low-Rank Adaptation) inserts small trainable matrices **A** and **B** into selected layers:

```
  h = W₀x  +  ΔWx         (standard fine-tuning)
  h = W₀x  +  BAx          (LoRA: B and A are small, W₀ is frozen)
```

With `r=8`, LoRA trains **< 0.5%** of parameters while achieving comparable performance.

In [21]:
try:
    from peft import LoraConfig, get_peft_model, TaskType

    base = AutoModelForCausalLM.from_pretrained("distilgpt2")

    lora_config = LoraConfig(
        task_type      = TaskType.CAUSAL_LM,
        r              = 8,         # Rank — higher = more capacity, more params
        lora_alpha     = 32,        # Scaling factor (usually 2–4× r)
        target_modules = ["c_attn"],# Attention projection layers
        lora_dropout   = 0.1,
        bias           = "none",
    )

    lora_model = get_peft_model(base, lora_config)

    total     = sum(p.numel() for p in lora_model.parameters())
    trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

    print(f"Total parameters     : {total:,}")
    print(f"Trainable (LoRA only): {trainable:,}")
    print(f"Trainable %          : {100 * trainable / total:.2f}%")
    print("\n✓ lora_model is ready — pass to Trainer exactly like a regular model")

except ImportError:
    print("peft not installed. Run: pip install peft")


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 14423.85it/s]

Total parameters     : 82,060,032
Trainable (LoRA only): 147,456
Trainable %          : 0.18%

✓ lora_model is ready — pass to Trainer exactly like a regular model


---
## Section 7 — Pipelines: One-Liner Inference

HuggingFace `pipeline()` wraps tokenisation + inference + decoding into a single object.


In [ ]:
# ── Text generation ──────────────────────────────────────────────────────────
gen = pipeline("text-generation", model="distilgpt2")
print(gen("Once upon a time", max_new_tokens=40, do_sample=True)[0]["generated_text"])

# ── Sentiment analysis (uses default model) ───────────────────────────────────
sent = pipeline("sentiment-analysis")
print(sent("This lab is incredibly well-structured!"))

# ── Translation ───────────────────────────────────────────────────────────────
# (requires Helsinki-NLP/opus-mt-en-fr — comment out if slow)
# trans = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr")
# print(trans("Artificial intelligence is fascinating."))


---
## Section 8 — Reflection Questions

Answer these in the markdown cells below (double-click to edit).

**Q1.** What is the difference between a Causal LM and a Seq2Seq model? Give one use-case for each.

**Answer:**

A Causal LM (e.g. GPT) predicts the *next* token, making it ideal for open-ended text generation. A Seq2Seq model (e.g. T5) encodes an input and decodes a separate output, making it ideal for translation, summarisation, and QA where the output structure differs from the input.

---
## Lab Complete!

### Next Steps
- Explore more models at [huggingface.co/models](https://huggingface.co/models)
- Try fine-tuning on a real dataset: `load_dataset("imdb")` or `load_dataset("squad")`
- Scale up with **Mistral-7B** or **LLaMA-3** using LoRA on Google Colab (free GPU)
- Add evaluation: `pip install evaluate` → BLEU, ROUGE, F1
- Deploy: wrap your model in FastAPI for a live QA endpoint
